# SCV-IDS Step 7 — Recover existing completed training (NO RETRAINING)

The original notebook finished 50 epochs but its scikit-learn t-SNE/embedding stage produced a native segmentation fault. We must preserve and reuse the already generated best checkpoint, training history, pre-crash validation predictions, split indices, and embedding vectors.

**Use this code in the ORIGINAL live Kaggle session**, not a fresh notebook with a different working directory. In the original Notebook 05, add the recovery cell **after** the failed training cell, and run **only this cell**. Do **not** choose "Run All" in Notebook 05, because it would rerun 50 training epochs. The new Notebook 06 is a versioned reference for this cell, not a standalone notebook to execute in a clean Kaggle session.

This approach changes the former unreliable t-SNE figures to **clearly marked PCA projections** from already saved vectors. Constant representations are explicitly marked degenerate; no artificially scattered points are generated. All 28 figures have PNG/SVG/PDF and CSV source artifacts in GitHub, while large checkpoints and individual validation probabilities remain in Kaggle.

**Prerequisites:** `/kaggle/working/research_outputs` still contains files from the terminated subprocess; your original Kaggle kernel/session must remain live. Enable Internet and `pushRare` secret access. The source dataset is not read, and no GPU is needed for recovery.

In [ ]:
# Run THIS CELL ALONE in your original Step-7 Kaggle notebook session.
from pathlib import Path
import os, subprocess, sys

REPO = Path("/kaggle/working/rare-attack-ids-step07")
OUT = Path("/kaggle/working/research_outputs/SCV-IDS-REPRO-001")
WEIGHTS = Path("/kaggle/working/research_outputs/checkpoints")

must_exist = [OUT/"training_history.csv", OUT/"validation_predictions.npz",
              OUT/"embedding_vectors.npz", OUT/"split_indices.npz",
              WEIGHTS/"scv_best.pt", WEIGHTS/"scv_last_resume.pt"]
for p in must_exist:
    print("SAVED" if p.is_file() else "MISSING", p)
if not all(p.is_file() for p in must_exist):
    raise FileNotFoundError("A required original training artifact is missing. DO NOT rerun training. Check the original Kaggle output/version.")

subprocess.run(["git","-C",str(REPO),"pull","--ff-only"],check=True)
print("Recovery commit:", subprocess.check_output(
    ["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip())

env=os.environ.copy()
env["PYTHONPATH"]=str(REPO/"src")
env["OPENBLAS_NUM_THREADS"]="1"
env["MKL_NUM_THREADS"]="1"
env["OMP_NUM_THREADS"]="1"
env["MPLBACKEND"]="Agg"
subprocess.run([sys.executable,str(REPO/"scripts"/"recover_scv_figures.py"),
                "--repo",str(REPO),"--output",str(OUT),
                "--checkpoints",str(WEIGHTS)],
               cwd=str(REPO),env=env,check=True)
print((OUT/"run_report.md").read_text(encoding="utf-8"))

In [ ]:
# Run after the recovery cell succeeds: synchronize observed sources and figures.
import os, subprocess, sys
try:
    from kaggle_secrets import UserSecretsClient
    token=UserSecretsClient().get_secret("pushRare")
except Exception:
    token=None
if not token:
    print("RECOVERED LOCALLY, NOT SYNCED: grant access to Kaggle secret pushRare.")
else:
    env=os.environ.copy()
    env["GITHUB_TOKEN"]=token
    try:
        subprocess.run([sys.executable,str(REPO/"scripts"/"sync_scv_results.py"),
                        "--run",str(OUT)],cwd=str(REPO),env=env,check=True)
    finally:
        env.pop("GITHUB_TOKEN",None)
        del token

In [ ]:
# Optional but strongly recommended: archive ALL original model evidence.
# Run in the same live Kaggle session; no model training.
from pathlib import Path
import subprocess, sys

ARCHIVE = Path("/kaggle/working/SCV-IDS-REPRO-001_FULL_ARTIFACTS.tar")
subprocess.run([
    sys.executable, str(REPO/"scripts"/"archive_scv_outputs.py"),
    "--base", "/kaggle/working/research_outputs",
    "--archive", str(ARCHIVE)
], cwd=str(REPO), check=True)
print("ARCHIVE READY:", ARCHIVE)
print("Download or persist this file before the Kaggle session expires.")


## Preservation and next action

Keep the original Kaggle session open until the sync succeeds. **GitHub only receives reproducible plotting CSVs, exported figures, metrics, configuration and checksums**. It does **not** contain `.pt`, `.npz`, or the raw dataset.

Run the archive cell and **download or persist** `/kaggle/working/SCV-IDS-REPRO-001_FULL_ARTIFACTS.tar` before the session expires. The archive contains all preserved checkpoints, raw predictions, embeddings, plot source tables and figures, plus a SHA-256 manifest; it is not uploaded to public GitHub. Avoid "Save & Run All" on the original Notebook 05 until its training cell has been disabled; running it would retrain. A dedicated **recovery-only** notebook/version may be used if it can access the saved artifacts. A GitHub commit is **not** a backup of large model checkpoints.

Send `VERIFIED SCV BASELINE SYNC COMMIT: ...`. If the recovery fails, send the first traceback and the `SAVED/MISSING` lines.